# 🏁 From Raw Timing Sheets to Race-Ready Data
## Notebook 1 — Data Cleaning & Feature Engineering

*Formula 1 World Championship, 1950–2024.* Before any exploratory analysis, the raw data has to be
**profiled, quality-checked, and enriched**. This notebook connects to the `f1_analytics` MySQL
database, audits it, and engineers three analysis-ready tables that Notebook 2 (the EDA) reads from.

**Pipeline**
1. Profile the raw tables (shape, dtypes)
2. Audit missing values & understand what they *mean* in F1 terms
3. Verify referential integrity (PK/FK)
4. Feature-engineer three enriched tables back into the database


## 🔌 Setup — SQLAlchemy connection

Everything runs live against MySQL 8.0 via SQLAlchemy + pandas.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
pd.set_option('display.max_columns', 40)

import os
_u=os.getenv('F1_DB_USER','root'); _p=os.getenv('F1_DB_PASSWORD','root')
_h=os.getenv('F1_DB_HOST','127.0.0.1'); _pt=os.getenv('F1_DB_PORT','3306'); _n=os.getenv('F1_DB_NAME','f1_analytics')
engine = create_engine(f'mysql+pymysql://{_u}:{_p}@{_h}:{_pt}/{_n}?charset=utf8mb4')
def q(sql):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

print('MySQL version:', q('SELECT VERSION() AS v').iloc[0,0])
q("SELECT COUNT(*) AS tables_in_db FROM information_schema.tables WHERE table_schema='f1_analytics'")


MySQL version: 8.0.45


,tables_in_db
0,19


## 1. 📸 Raw data profiling — shape of every table

The dataset is a classic star-ish schema: dimension tables (drivers, constructors, circuits,
seasons, status) around the central `races`, with fact tables (results, qualifying, lap_times,
pit_stops, standings) hanging off them.

In [2]:
tables = ['circuits','constructors','drivers','seasons','status','races','qualifying',
          'pit_stops','lap_times','results','sprint_results','driver_standings',
          'constructor_standings','constructor_results']
rows = [(t, q(f'SELECT COUNT(*) n FROM {t}').iloc[0,0]) for t in tables]
shape = pd.DataFrame(rows, columns=['table','row_count']).sort_values('row_count', ascending=False)
shape.reset_index(drop=True)


,table,row_count
0,lap_times,589081
1,driver_standings,34863
2,results,26759
3,constructor_standings,13391
4,constructor_results,12625
5,pit_stops,11371
6,qualifying,10494
7,races,1125
8,drivers,861
9,sprint_results,360


In [3]:
# Column layout of the central fact table `results`
q('''SELECT COLUMN_NAME, DATA_TYPE, IS_NULLABLE, COLUMN_KEY
     FROM information_schema.columns
     WHERE table_schema='f1_analytics' AND table_name='results'
     ORDER BY ORDINAL_POSITION''')


,COLUMN_NAME,DATA_TYPE,IS_NULLABLE,COLUMN_KEY
0,resultId,int,NO,PRI
1,raceId,int,YES,MUL
2,driverId,int,YES,MUL
3,constructorId,int,YES,MUL
4,number,int,YES,
5,grid,int,YES,MUL
6,position,int,YES,
7,positionText,varchar,YES,
8,positionOrder,int,YES,
9,points,float,YES,


## 2. 🧹 Missing-value audit — and what NULL *means* here

In F1 data, missing is not always "dirty". Three cases matter:
- `results.position IS NULL` → driver **did not finish (DNF)** — a real racing outcome, not bad data.
- `results.milliseconds IS NULL` → no total race time (DNF, or lapped/older races).
- `results.grid = 0` → started from the **pit lane** (no grid slot), not a missing value.

The Kaggle CSVs encode unknowns as `\N`, which loaded cleanly as SQL `NULL`.

In [4]:
audit = q('''
SELECT 'results.position (DNF)'  AS field, SUM(position IS NULL)     AS missing, COUNT(*) total FROM results
UNION ALL SELECT 'results.milliseconds', SUM(milliseconds IS NULL), COUNT(*) FROM results
UNION ALL SELECT 'results.grid = 0 (pit start)', SUM(grid = 0),      COUNT(*) FROM results
UNION ALL SELECT 'drivers.dob',           SUM(dob IS NULL),          COUNT(*) FROM drivers
UNION ALL SELECT 'drivers.number',        SUM(number IS NULL),       COUNT(*) FROM drivers
UNION ALL SELECT 'qualifying.q3',         SUM(q3 IS NULL OR q3=''),  COUNT(*) FROM qualifying''')
audit['missing_pct'] = (100*audit['missing']/audit['total']).round(1)
audit


,field,missing,total,missing_pct
0,results.position (DNF),10953.0,26759,40.9
1,results.milliseconds,19079.0,26759,71.3
2,results.grid = 0 (pit start),1638.0,26759,6.1
3,drivers.dob,0.0,861,0.0
4,drivers.number,802.0,861,93.1
5,qualifying.q3,6865.0,10494,65.4


## 3. 🔗 Referential integrity — PK/FK sanity

Every foreign key should point at a real parent row. All counts below must be **zero**.

In [5]:
q('''
SELECT 'results→races'   AS relationship, COUNT(*) orphans FROM results r  LEFT JOIN races   x ON x.raceId=r.raceId       WHERE x.raceId IS NULL
UNION ALL SELECT 'results→drivers',   COUNT(*) FROM results r LEFT JOIN drivers x ON x.driverId=r.driverId WHERE x.driverId IS NULL
UNION ALL SELECT 'results→constructors', COUNT(*) FROM results r LEFT JOIN constructors x ON x.constructorId=r.constructorId WHERE x.constructorId IS NULL
UNION ALL SELECT 'results→status',    COUNT(*) FROM results r LEFT JOIN status x ON x.statusId=r.statusId WHERE x.statusId IS NULL
UNION ALL SELECT 'lap_times→races',   COUNT(*) FROM lap_times l LEFT JOIN races x ON x.raceId=l.raceId WHERE x.raceId IS NULL''')


,relationship,orphans
0,results→races,0
1,results→drivers,0
2,results→constructors,0
3,results→status,0
4,lap_times→races,0


## 4. ⚙️ Feature engineering

Raw rows don't answer questions directly. We derive three enriched tables **inside MySQL** so the EDA
notebook can query them straight away:

| Table | Grain | Key engineered features |
|-------|-------|------------------------|
| `f1_result_features`  | one row per race result | `is_win`, `is_podium`, `is_dnf`, `positions_gained`, `driver_age`, `decade`, `status_category` |
| `f1_driver_features`  | one row per driver | `starts`, `wins`, `podiums`, `poles`, `win_pct`, `dnf_rate`, `avg_grid`, `teams`, `career_years` |
| `f1_season_driver`    | one row per driver-season | `wins`, `dnfs`, `points`, `avg_grid`, `avg_finish` |

`status_category` buckets the 139 raw status codes into **Finished / Lapped / Mechanical /
Accident-Driver / DNS-DNQ / Other**, using exactly the mechanical-failure list from
`10_capstone_thesis.sql` so the notebook, the dashboard and the findings report all agree — the single most useful cleaning step for reliability analysis.

In [6]:
# 4a. Result-level features (one row per race entry)
ddl_result = '''
DROP TABLE IF EXISTS f1_result_features;
CREATE TABLE f1_result_features AS
SELECT
  re.resultId, re.raceId, ra.year,
  FLOOR(ra.year/10)*10                               AS decade,
  re.driverId, re.constructorId,
  TIMESTAMPDIFF(YEAR, d.dob, ra.date)                AS driver_age,
  re.grid, re.positionOrder, re.points,
  (re.positionOrder = 1)                             AS is_win,
  (re.positionOrder <= 3)                            AS is_podium,
  (re.position IS NULL)                              AS is_dnf,
  CASE WHEN re.grid > 0 THEN re.grid - re.positionOrder END AS positions_gained,
  s.status,
  CASE
    WHEN s.status = 'Finished'                       THEN 'Finished'
    WHEN s.status LIKE '%Lap%'                        THEN 'Lapped'
    WHEN s.status IN ('Accident','Collision','Spun off','Collision damage',
                      'Puncture','Damage','Driver unwell','Injury','Fatal accident') THEN 'Accident/Driver'
    WHEN s.status LIKE '%Did not%' OR s.status IN ('Withdrew','Not classified',
                      'Injured','Excluded','Disqualified') THEN 'DNS/DNQ'
    WHEN s.status IN ('Engine','Gearbox','Transmission','Hydraulics','Electrical',
         'Suspension','Brakes','Clutch','Overheating','Power Unit','Fuel system',
         'Oil leak','Water leak','Driveshaft','Radiator','Throttle','Turbo','Exhaust',
         'Differential','Alternator','Fuel pump','Ignition','Oil pressure','Wheel',
         'Halfshaft','Mechanical','Power loss','Fuel leak','Fuel pressure',
         'Cooling system','Vibrations','ERS','Battery','Distributor','Pneumatics',
         'Engine fire','Spark plugs','Wheel bearing','Oil line') THEN 'Mechanical'
    ELSE 'Other'
  END                                                AS status_category
FROM results re
JOIN races   ra ON ra.raceId = re.raceId
JOIN drivers d  ON d.driverId = re.driverId
JOIN status  s  ON s.statusId = re.statusId;'''
with engine.begin() as conn:
    for stmt in ddl_result.strip().split(';'):
        if stmt.strip(): conn.execute(text(stmt))
q('SELECT * FROM f1_result_features LIMIT 5')


,resultId,raceId,year,decade,driverId,constructorId,driver_age,grid,positionOrder,points,is_win,is_podium,is_dnf,positions_gained,status,status_category
0,1,18,2008,2000,1,1,23,1,1,10.0,1,1,0,0,Finished,Finished
1,27,19,2008,2000,1,1,23,9,5,4.0,0,0,0,4,Finished,Finished
2,57,20,2008,2000,1,1,23,3,13,0.0,0,0,0,-10,+1 Lap,Lapped
3,69,21,2008,2000,1,1,23,5,3,6.0,0,1,0,2,Finished,Finished
4,90,22,2008,2000,1,1,23,3,2,8.0,0,1,0,1,Finished,Finished


In [7]:
# 4b. Driver career features (one row per driver)
ddl_driver = '''
DROP TABLE IF EXISTS f1_driver_features;
CREATE TABLE f1_driver_features AS
SELECT
  d.driverId,
  CONCAT(d.forename,' ',d.surname) AS driver,
  d.nationality,
  COUNT(*)                         AS starts,
  SUM(f.is_win)                    AS wins,
  SUM(f.is_podium)                 AS podiums,
  SUM(f.grid = 1)                  AS poles,
  ROUND(SUM(f.points),1)           AS points,
  ROUND(100*SUM(f.is_win)/COUNT(*),2)  AS win_pct,
  ROUND(100*SUM(f.is_dnf)/COUNT(*),2)  AS dnf_rate,
  ROUND(AVG(NULLIF(f.grid,0)),2)   AS avg_grid,
  COUNT(DISTINCT f.constructorId)  AS teams,
  MIN(f.year)                      AS first_season,
  MAX(f.year)                      AS last_season,
  MAX(f.year)-MIN(f.year)+1        AS career_years
FROM f1_result_features f
JOIN drivers d ON d.driverId = f.driverId
GROUP BY d.driverId;'''
with engine.begin() as conn:
    for stmt in ddl_driver.strip().split(';'):
        if stmt.strip(): conn.execute(text(stmt))
q('SELECT * FROM f1_driver_features ORDER BY wins DESC LIMIT 5')


,driverId,driver,nationality,starts,wins,podiums,poles,points,win_pct,dnf_rate,avg_grid,teams,first_season,last_season,career_years
0,1,Lewis Hamilton,British,356,105.0,202.0,104.0,4820.5,29.49,8.43,4.31,2,2007,2024,18
1,30,Michael Schumacher,German,308,91.0,155.0,68.0,1566.0,29.55,21.75,4.87,4,1991,2012,22
2,830,Max Verstappen,Dutch,209,63.0,112.0,40.0,2912.5,30.14,14.35,4.98,2,2015,2024,10
3,20,Sebastian Vettel,German,300,53.0,122.0,57.0,3098.0,17.67,12.67,6.31,5,2007,2022,16
4,117,Alain Prost,French,202,51.0,106.0,33.0,798.5,25.25,29.21,4.16,4,1980,1993,14


In [8]:
# 4c. Season-by-driver features (one row per driver per season)
ddl_season = '''
DROP TABLE IF EXISTS f1_season_driver;
CREATE TABLE f1_season_driver AS
SELECT
  f.year, f.driverId,
  COUNT(*)               AS races,
  SUM(f.is_win)          AS wins,
  SUM(f.is_podium)       AS podiums,
  SUM(f.is_dnf)          AS dnfs,
  ROUND(SUM(f.points),1) AS points,
  ROUND(AVG(NULLIF(f.grid,0)),2)     AS avg_grid,
  ROUND(AVG(f.positionOrder),2)      AS avg_finish
FROM f1_result_features f
GROUP BY f.year, f.driverId;'''
with engine.begin() as conn:
    for stmt in ddl_season.strip().split(';'):
        if stmt.strip(): conn.execute(text(stmt))
q('SELECT * FROM f1_season_driver WHERE year=2021 ORDER BY points DESC LIMIT 5')


,year,driverId,races,wins,podiums,dnfs,points,avg_grid,avg_finish
0,2021,830,22,10.0,18.0,3.0,388.5,2.86,4.14
1,2021,1,22,8.0,17.0,1.0,385.5,3.09,3.41
2,2021,822,22,1.0,11.0,4.0,219.0,5.59,7.41
3,2021,815,22,1.0,5.0,2.0,190.0,5.39,7.55
4,2021,832,22,0.0,4.0,0.0,163.5,7.91,6.50


## ✅ Engineered tables ready

Confirm the three new tables exist with sensible row counts, then hand off to **Notebook 2 — EDA**.

In [9]:
q('''SELECT table_name, table_rows
     FROM information_schema.tables
     WHERE table_schema='f1_analytics' AND table_name LIKE 'f1_%'
     ORDER BY table_name''')


,TABLE_NAME,TABLE_ROWS
0,f1_driver_features,861
1,f1_result_features,26690
2,f1_season_driver,3211


**Status-category distribution** — a first look at what these features unlock (full analysis in Notebook 2):

In [10]:
q('''SELECT status_category, COUNT(*) AS entries,
         ROUND(100*COUNT(*)/SUM(COUNT(*)) OVER (),1) AS pct
     FROM f1_result_features
     GROUP BY status_category ORDER BY entries DESC''')


,status_category,entries,pct
0,Finished,7674,28.7
1,Lapped,7487,28.0
2,Mechanical,6137,22.9
3,Accident/Driver,2840,10.6
4,DNS/DNQ,1938,7.2
5,Other,683,2.6
